<a href="https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/VIX_CALIBRATED_THRESHOLD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# VIX Calibrated Threshold v1 — cibler le point faible UP_FORT/DOWN_FORT (7/7)

**Rôle.** Nouveau notebook indépendant, à lancer après `VIX_FINAL_FEATURES`. Ne change ni les
features ni le modèle : il s'attaque directement au **point faible identifié et répété dans
tout ce projet** — `F1_dir` tourne autour de 0.61 alors que `F1_UP_FORT`/`F1_DOWN_FORT` plafonnent
à 0.36/0.63. Toutes les expériences précédentes (stacking, spike features, particle filter,
routing, TFT) ont cherché à améliorer ce point via de nouvelles features/architectures — sans
succès en walk-forward. Ce notebook essaie un levier différent, jamais testé : la **décision**
elle-même.

**Ce qui est testé** :
1. **RAW** (référence) : le classifieur (RandomForest / XGBoost) tel qu'utilisé partout ailleurs
   — décision par argmax sur les 4 classes.
2. **CALIBRATED** : mêmes features/mêmes données, mais (a) les probabilités sont recalibrées par
   régression isotonique sur une tranche de validation prélevée **chronologiquement** à la fin du
   train (donc toujours causale, jamais sur le test), et (b) le seuil de décision FORT/FAIBLE au
   sein de la direction prédite (UP ou DOWN) est recherché sur cette même tranche de validation
   (grille de 0.30 à 0.70), au lieu du seuil implicite à 0.50 de l'argmax.

**Pourquoi ça peut aider** : un classifieur multi-classe entraîné avec un objectif macro peut être
mal calibré sur la sous-tâche FORT vs FAIBLE (minoritaire dans chaque direction). Recalibrer et
ajuster le seuil ne change rien au modèle ni aux features — seulement la façon de convertir ses
probabilités en décision — donc c'est le test le moins coûteux et le plus ciblé qu'il reste à
essayer sur ce point faible précis.

**Comparateurs déjà établis** (walk-forward) : GLOBAL RandomForest h=5j (F1_dir≈0.610±0.025,
F1_UP_FORT≈0.359, F1_DOWN_FORT≈0.627).


In [ ]:
import subprocess, sys
pkgs = ['xgboost', 'shap', 'xlsxwriter', 'imbalanced-learn', 'pyarrow']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


In [ ]:
import os, time, json, warnings, random
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import f1_score, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_CALIBRATED_THRESHOLD'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'flat_thr': 0.003,
    'horizons': [1, 2, 3, 5, 7, 10],
    'regimes': ['GLOBAL', 'CALM', 'NORMAL', 'STRESS'],
    'algos': ['RandomForest', 'XGBoost'],
    'N': 8,                  # point de fonctionnement établi
    'sampler': 'SMOTE',
    'n_wf_folds': 5,
    'min_train_frac': 0.40,   # doit matcher VIX_FINAL_FEATURES
    'val_frac': 0.20,         # fraction finale du train utilisée pour calibrer + choisir le seuil
    'thr_grid': [round(x, 2) for x in np.arange(0.30, 0.71, 0.05)],
    'shap_sample': 500,
    'pool_prefilter': 450,
    'min_train_rows': 120, 'min_val_rows': 20, 'min_test_rows': 20,
}
TARGET_COL = 'VIX_Amplitude_Class'
RESULTS_CSV = 'vix_calibrated_threshold_results.csv'
GITHUB_REPO = 'LP-D/claude'
FEATURES_BRANCH = 'results/vix-final-features'
RESULTS_BRANCH = 'results/vix-calibrated-threshold'

n_combos = len(CONFIG['horizons']) * len(CONFIG['regimes']) * len(CONFIG['algos']) * CONFIG['n_wf_folds']
print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | N={CONFIG['N']} sampler={CONFIG['sampler']} "
      f"val_frac={CONFIG['val_frac']} | grille = {n_combos} lignes (chacune produit RAW + CALIBRATED) "
      f"({len(CONFIG['horizons'])}h × {len(CONFIG['regimes'])}reg × {len(CONFIG['algos'])}algos × "
      f"{CONFIG['n_wf_folds']}folds)")


In [ ]:
# ============================================================
# CHARGEMENT DU DATASET PARTAGÉ (produit par VIX_FINAL_FEATURES)
# ============================================================
import subprocess

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

if not os.path.exists('vix_final_features.parquet'):
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    url = f"https://{auth}github.com/{GITHUB_REPO}.git"
    workdir = "/content/_vix_features_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", FEATURES_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(
            "Impossible de récupérer le dataset partagé depuis "
            f"'{FEATURES_BRANCH}'. As-tu bien exécuté VIX_FINAL_FEATURES.ipynb en premier "
            f"(et poussé son résultat) ? Détail: {clone.stderr[-500:]}")
    subprocess.run(["cp", f"{workdir}/vix_final_features.parquet", "."], check=True)
    subprocess.run(["cp", f"{workdir}/vix_final_features_meta.json", "."], check=True)
    print(f"[PULL OK] Dataset récupéré depuis '{FEATURES_BRANCH}'")
else:
    print("[SKIP] vix_final_features.parquet déjà présent localement")

df_features = pd.read_parquet('vix_final_features.parquet')
with open('vix_final_features_meta.json') as f:
    meta = json.load(f)
FEATURE_POOL = meta['feature_pool']
VIX_COL = meta['vix_col']; SPX_COL = meta['spx_col']
print(f"Dataset: {df_features.shape} | VIX={VIX_COL} | pool: {len(FEATURE_POOL)} features "
      f"| source: {meta['date_min']} → {meta['date_max']}")

all_dates = df_features.dropna(how='all').index.sort_values()
n_obs = len(all_dates)
first_cut = int(n_obs * CONFIG['min_train_frac'])
test_span = (n_obs - first_cut) // CONFIG['n_wf_folds']
FOLD_CUTS = [first_cut + k * test_span for k in range(CONFIG['n_wf_folds'] + 1)]
FOLD_CUTS[-1] = n_obs
for k in range(CONFIG['n_wf_folds']):
    print(f"  Fold {k+1}: train → {all_dates[FOLD_CUTS[k]-1].date()} | "
          f"test {all_dates[FOLD_CUTS[k]].date()} → {all_dates[FOLD_CUTS[k+1]-1].date()}")


In [ ]:
def build_target(vix_series, horizon, split_idx):
    vix = vix_series.ffill().bfill(); vix_tr = vix.iloc[:split_idx]
    calm_thr = vix_tr.quantile(0.33); stress_thr = vix_tr.quantile(0.67)
    regime = pd.Series('NORMAL', index=vix.index)
    regime[vix < calm_thr] = 'CALM'; regime[vix >= stress_thr] = 'STRESS'
    ret = (vix.shift(-horizon) / vix) - 1
    flat = ret.abs() < CONFIG['flat_thr']
    ret = ret.loc[~flat].dropna(); reg_r = regime.reindex(ret.index)
    cut_date = vix.index[min(split_idx, len(vix) - 1)]
    ret_tr = ret.loc[ret.index < cut_date]; reg_tr = reg_r.loc[ret_tr.index]
    thr = {}
    for reg in ['CALM', 'NORMAL', 'STRESS']:
        sub = ret_tr[reg_tr == reg]
        thr[reg] = (sub.quantile(0.25) if len(sub) >= 20 else ret_tr.quantile(0.25),
                    sub.quantile(0.75) if len(sub) >= 20 else ret_tr.quantile(0.75))
    thr['GLOBAL'] = (ret_tr.quantile(0.25), ret_tr.quantile(0.75))
    def classify(r, reg):
        q25, q75 = thr.get(reg, (0, 0))
        if r < q25: return 0
        if r < 0:   return 1
        if r < q75: return 2
        return 3
    target = pd.Series([classify(r, reg_r[i]) for i, r in ret.items()], index=ret.index, name=TARGET_COL)
    return target, reg_r, thr

def metrics(y_true, y_pred):
    dm = {0: 'DOWN', 1: 'DOWN', 2: 'UP', 3: 'UP'}
    yd_t = [dm[y] for y in y_true]; yd_p = [dm[y] for y in y_pred]
    m = {'F1_4cls': round(f1_score(y_true, y_pred, average='macro', zero_division=0), 4),
         'Acc_dir': round(accuracy_score(yd_t, yd_p), 4),
         'F1_dir': round(f1_score(yd_t, yd_p, average='macro', zero_division=0), 4)}
    ui = [i for i, y in enumerate(y_true) if dm[y] == 'UP']
    di = [i for i, y in enumerate(y_true) if dm[y] == 'DOWN']
    if len(ui) >= 10:
        yt = ['FORT' if y_true[i] == 3 else 'FAIBLE' for i in ui]
        yp = ['FORT' if y_pred[i] == 3 else 'FAIBLE' for i in ui]
        m['F1_UP_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_UP_FORT'] = np.nan
    if len(di) >= 10:
        yt = ['FORT' if y_true[i] == 0 else 'FAIBLE' for i in di]
        yp = ['FORT' if y_pred[i] == 0 else 'FAIBLE' for i in di]
        m['F1_DOWN_FORT'] = round(f1_score(yt, yp, pos_label='FORT', average='binary', zero_division=0), 4)
    else: m['F1_DOWN_FORT'] = np.nan
    return m

def decide(proba, thr):
    """Décision hiérarchique : direction (UP/DOWN) puis FORT/FAIBLE au seuil `thr`.
    Classes : 0=DOWN_FORT, 1=DOWN_faible, 2=UP_faible, 3=UP_FORT."""
    p_down = proba[:, 0] + proba[:, 1]; p_up = proba[:, 2] + proba[:, 3]
    up_fort_ratio = np.divide(proba[:, 3], p_up, out=np.zeros_like(p_up), where=p_up > 1e-12)
    down_fort_ratio = np.divide(proba[:, 0], p_down, out=np.zeros_like(p_down), where=p_down > 1e-12)
    pred = np.where(p_up >= p_down,
                    np.where(up_fort_ratio >= thr, 3, 2),
                    np.where(down_fort_ratio >= thr, 0, 1))
    return pred.astype(int)

def get_clf(algo):
    if algo == 'XGBoost':
        return XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, min_child_weight=3, eval_metric='mlogloss',
                             objective='multi:softprob', random_state=SEED, n_jobs=-1, verbosity=0)
    if algo == 'RandomForest':
        return RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=5,
                                      class_weight='balanced', random_state=SEED, n_jobs=-1)
    raise ValueError(algo)

def get_samp(name):
    return {'SMOTE': SMOTE(random_state=SEED)}[name]

def shap_rank(X_tr, y_tr, pool_names, top_n, prefilter):
    nf = X_tr.shape[1]
    if nf > prefilter:
        pf = XGBClassifier(n_estimators=60, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                           eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
        pf.fit(X_tr, y_tr); keep = np.argsort(pf.feature_importances_)[::-1][:prefilter]
    else:
        keep = np.arange(nf)
    Xk = X_tr[:, keep]
    pilot = XGBClassifier(n_estimators=80, max_depth=4, learning_rate=0.1, objective='multi:softprob',
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1, verbosity=0)
    pilot.fit(Xk, y_tr)
    sv = np.abs(np.array(shap.TreeExplainer(pilot).shap_values(Xk[:min(CONFIG['shap_sample'], len(Xk))])))
    nfk = Xk.shape[1]
    feat_axes = [ax for ax in range(sv.ndim) if sv.shape[ax] == nfk]
    if len(feat_axes) == 1:
        arr = sv.mean(axis=tuple(ax for ax in range(sv.ndim) if ax != feat_axes[0]))
    else:
        arr = np.asarray(pilot.feature_importances_)
    order = np.argsort(np.asarray(arr).ravel())[::-1][:top_n]
    return list(keep[order])

print("Helpers OK (build_target, metrics, decide, get_clf, get_samp, shap_rank)")


## Principe : calibration de probabilités + seuil de décision causal

Un classifieur entraîné à minimiser une perte multi-classe (log-loss / Gini) n'a aucune garantie
que ses probabilités de sortie soient **calibrées** — c'est-à-dire que parmi toutes les
prédictions où il annonce "70% de chances", la fréquence réelle observée soit bien 70%. C'est
particulièrement vrai pour les classes minoritaires comme FORT (au sein d'UP ou de DOWN), sur
lesquelles le rééchantillonnage (SMOTE) déforme encore les probabilités brutes.

**Recalibration isotonique** : on apprend une fonction monotone qui corrige les probabilités
brutes du modèle pour qu'elles correspondent mieux aux fréquences observées, à partir d'un
échantillon **jamais vu pendant l'entraînement** du modèle de base. Ici, cet échantillon est une
tranche de validation prélevée **chronologiquement en fin de train** (jamais le test), pour rester
causal — le modèle de base est entraîné sur le train amputé de cette tranche, puis la calibration
et la recherche de seuil se font sur cette tranche, avant d'être appliquées telles quelles au test.

**Seuil de décision** : la métrique hiérarchique de ce projet décompose la prédiction en deux
étapes — direction (UP/DOWN) puis FORT/FAIBLE au sein de la direction. L'argmax standard revient à
un seuil implicite de 0.50 sur cette deuxième étape. Rien ne garantit que 0.50 soit optimal pour
`F1_UP_FORT`/`F1_DOWN_FORT` spécifiquement : ce notebook recherche, sur la tranche de validation
causale, le seuil (dans {0.30, 0.35, ..., 0.70}) qui maximise la moyenne de ces deux métriques,
puis l'applique tel quel au test — sans jamais optimiser directement sur les données de test.


In [ ]:
# ============================================================
# SYNCHRONISATION DE LA PROGRESSION (même principe que les autres notebooks)
# ============================================================
_PUSH_WORKDIR = "/content/_vix_calib_push"

def push_progress(label=''):
    if not GITHUB_TOKEN or not os.path.exists(RESULTS_CSV):
        return False
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0: return False
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        subprocess.run(["cp", RESULTS_CSV, f"{_PUSH_WORKDIR}/{RESULTS_CSV}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email", "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name", "VIX Calibrated Threshold Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", RESULTS_CSV], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Progression Calibrated Threshold {label} — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        ok = push.returncode == 0
        if ok: print(f"  [CHECKPOINT PUSHÉ] {label}")
        return ok
    except Exception as e:
        print(f"  [WARN push checkpoint] {e}")
        return False

def pull_progress():
    if os.path.exists(RESULTS_CSV) or not GITHUB_TOKEN:
        return
    url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    exists = subprocess.run(["git", "ls-remote", "--exit-code", "--heads", url, RESULTS_BRANCH],
                            capture_output=True, text=True)
    if exists.returncode != 0:
        print(f"[INFO] Aucune progression antérieure sur '{RESULTS_BRANCH}'.")
        return
    workdir = "/content/_vix_calib_pull"
    subprocess.run(["rm", "-rf", workdir], check=False)
    clone = subprocess.run(["git", "clone", "--depth", "1", "--branch", RESULTS_BRANCH, url, workdir],
                           capture_output=True, text=True)
    if clone.returncode == 0 and os.path.exists(f"{workdir}/{RESULTS_CSV}"):
        subprocess.run(["cp", f"{workdir}/{RESULTS_CSV}", "."], check=True)
        print("[PULL OK] Progression Calibrated Threshold antérieure récupérée.")

pull_progress()


In [ ]:
# ============================================================
# RAW (protocole établi) vs CALIBRATED (calibration + seuil causal appris sur une
# tranche de validation prélevée en fin de train)
# ============================================================
KEY_COLS = ['horizon', 'regime', 'algo', 'fold']

done_keys = set()
if os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0:
    prev = pd.read_csv(RESULTS_CSV, usecols=KEY_COLS)
    done_keys = set(map(tuple, prev.values.tolist()))
    print(f"[REPRISE] {len(done_keys)} lignes déjà calculées.")

def save_row(row):
    header = not (os.path.exists(RESULTS_CSV) and os.path.getsize(RESULTS_CSV) > 0)
    pd.DataFrame([row]).to_csv(RESULTS_CSV, mode='a', header=header, index=False)
    done_keys.add(tuple(row[c] for c in KEY_COLS))

t0 = time.time()
n_saved = 0
for h in CONFIG['horizons']:
    for reg in CONFIG['regimes']:
        for k in range(CONFIG['n_wf_folds']):
            for algo in CONFIG['algos']:
                key = (h, reg, algo, k)
                if key in done_keys:
                    continue
                cut, nxt = FOLD_CUTS[k], FOLD_CUTS[k + 1]
                cut_date, nxt_date = all_dates[cut], all_dates[nxt - 1]
                target, reg_r, _ = build_target(df_features[VIX_COL], h, cut)
                idx = target.index
                tr_mask_full = np.asarray(idx < cut_date)
                te_mask = np.asarray((idx >= cut_date) & (idx <= nxt_date))
                if reg != 'GLOBAL':
                    reg_al = reg_r.reindex(idx).fillna('NORMAL').values
                    tr_mask_full = tr_mask_full & (reg_al == reg); te_mask = te_mask & (reg_al == reg)

                tr_positions = np.where(tr_mask_full)[0]
                n_tr = len(tr_positions)
                y_te = target.values[te_mask].astype(int)
                if n_tr < CONFIG['min_train_rows'] or len(y_te) < CONFIG['min_test_rows']:
                    continue
                n_val = max(CONFIG['min_val_rows'], int(n_tr * CONFIG['val_frac']))
                if n_val >= n_tr or (n_tr - n_val) < CONFIG['min_train_rows'] // 2:
                    continue
                n_fit = n_tr - n_val

                X_pool = df_features[FEATURE_POOL].reindex(idx)
                sc = RobustScaler()
                X_tr_full = sc.fit_transform(np.nan_to_num(X_pool.values[tr_positions]))
                X_te = sc.transform(np.nan_to_num(X_pool.values[te_mask]))
                y_tr_full = target.values[tr_positions].astype(int)
                X_fit, y_fit = X_tr_full[:n_fit], y_tr_full[:n_fit]
                X_val, y_val = X_tr_full[n_fit:], y_tr_full[n_fit:]

                fidx = shap_rank(X_tr_full, y_tr_full, FEATURE_POOL, CONFIG['N'], CONFIG['pool_prefilter'])

                # --- RAW : protocole établi (train complet, argmax) ---
                try:
                    Xr, yr = get_samp(CONFIG['sampler']).fit_resample(X_tr_full[:, fidx], y_tr_full)
                except Exception:
                    Xr, yr = X_tr_full[:, fidx], y_tr_full
                clf_raw = get_clf(algo); clf_raw.fit(Xr, yr)
                met_raw = metrics(y_te, clf_raw.predict(X_te[:, fidx]))

                # --- CALIBRATED : base sur fit seul, calibrée + seuil choisi sur val (causal) ---
                try:
                    Xr_fit, yr_fit = get_samp(CONFIG['sampler']).fit_resample(X_fit[:, fidx], y_fit)
                except Exception:
                    Xr_fit, yr_fit = X_fit[:, fidx], y_fit
                clf_base = get_clf(algo); clf_base.fit(Xr_fit, yr_fit)
                try:
                    calib = CalibratedClassifierCV(clf_base, method='isotonic', cv='prefit')
                    calib.fit(X_val[:, fidx], y_val)
                    proba_val = calib.predict_proba(X_val[:, fidx])
                    proba_te = calib.predict_proba(X_te[:, fidx])
                except Exception:
                    proba_val = clf_base.predict_proba(X_val[:, fidx])
                    proba_te = clf_base.predict_proba(X_te[:, fidx])

                best_thr, best_score = 0.5, -1.0
                for thr in CONFIG['thr_grid']:
                    m_val = metrics(y_val, decide(proba_val, thr))
                    ups = m_val['F1_UP_FORT'] if not np.isnan(m_val['F1_UP_FORT']) else 0.0
                    dns = m_val['F1_DOWN_FORT'] if not np.isnan(m_val['F1_DOWN_FORT']) else 0.0
                    score = (ups + dns) / 2
                    if score > best_score:
                        best_score, best_thr = score, thr

                met_cal = metrics(y_te, decide(proba_te, best_thr))

                save_row({'horizon': h, 'regime': reg, 'algo': algo, 'fold': k,
                          'n_train': n_tr, 'n_fit': n_fit, 'n_val': n_val, 'n_test': len(y_te),
                          'best_thr': best_thr, 'val_score': round(best_score, 4),
                          **{f'RAW_{kk}': v for kk, v in met_raw.items()},
                          **{f'CAL_{kk}': v for kk, v in met_cal.items()}})
                n_saved += 1
                if n_saved % 50 == 0:
                    print(f"  ... {len(done_keys)}/{n_combos} lignes | {(time.time()-t0)/60:.1f}min")
                    push_progress(label=f"{len(done_keys)}/{n_combos}")

print(f"\n[CALIBRATED THRESHOLD] {len(done_keys)}/{n_combos} lignes calculées ({(time.time()-t0)/60:.1f}min)")
push_progress(label='fin de run')


In [ ]:
# ============================================================
# SYNTHÈSE : CALIBRATED vs RAW
# ============================================================
df_c = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
print(f"Progression: {len(df_c)}/{n_combos} ({len(df_c)/max(n_combos,1):.1%})")

if len(df_c):
    ok = df_c.dropna(subset=['RAW_F1_dir', 'CAL_F1_dir'])
    ok['delta_F1_dir'] = (ok['CAL_F1_dir'] - ok['RAW_F1_dir']).round(4)
    ok['delta_F1_UP_FORT'] = (ok['CAL_F1_UP_FORT'] - ok['RAW_F1_UP_FORT']).round(4)
    ok['delta_F1_DOWN_FORT'] = (ok['CAL_F1_DOWN_FORT'] - ok['RAW_F1_DOWN_FORT']).round(4)

    agg = (ok.groupby(['horizon', 'regime', 'algo'])
           .agg(RAW_F1_dir=('RAW_F1_dir', 'mean'), CAL_F1_dir=('CAL_F1_dir', 'mean'),
                delta_F1_dir=('delta_F1_dir', 'mean'),
                RAW_F1_UP_FORT=('RAW_F1_UP_FORT', 'mean'), CAL_F1_UP_FORT=('CAL_F1_UP_FORT', 'mean'),
                delta_F1_UP_FORT=('delta_F1_UP_FORT', 'mean'),
                RAW_F1_DOWN_FORT=('RAW_F1_DOWN_FORT', 'mean'), CAL_F1_DOWN_FORT=('CAL_F1_DOWN_FORT', 'mean'),
                delta_F1_DOWN_FORT=('delta_F1_DOWN_FORT', 'mean'),
                n_folds=('fold', 'nunique'))
           .reset_index().round(4))
    print("\n### CALIBRATED vs RAW, moyenne walk-forward par (horizon, régime, algo) ###")
    print(agg.sort_values('delta_F1_UP_FORT', ascending=False).to_string(index=False))

    print(f"\nDelta F1_dir moyen : {ok['delta_F1_dir'].mean():+.4f}")
    print(f"Delta F1_UP_FORT moyen : {ok['delta_F1_UP_FORT'].mean():+.4f}")
    print(f"Delta F1_DOWN_FORT moyen : {ok['delta_F1_DOWN_FORT'].mean():+.4f}")
    print(f"Seuil retenu (médian sur toutes les configs) : {ok['best_thr'].median():.2f} "
          f"(0.50 = argmax standard)")

    best_row = agg.sort_values('CAL_F1_UP_FORT', ascending=False).iloc[0]
    print(f"\nMeilleure config calibrée : h={best_row['horizon']}j {best_row['regime']} {best_row['algo']} "
          f"→ F1_UP_FORT={best_row['CAL_F1_UP_FORT']:.4f} (RAW: {best_row['RAW_F1_UP_FORT']:.4f})")

    print("\nRéférence établie (walk-forward) : GLOBAL RandomForest h=5j "
          "F1_dir=0.610±0.025  F1_UP_FORT=0.359  F1_DOWN_FORT=0.627")

    if ok['delta_F1_UP_FORT'].mean() > 0.01 or ok['delta_F1_DOWN_FORT'].mean() > 0.01:
        print("\n[VERDICT] La calibration + seuil causal améliore en moyenne le point faible "
              "identifié (F1_UP_FORT/F1_DOWN_FORT) sans changer ni features ni modèle — à adopter "
              "comme réglage de décision par défaut, en gardant F1_dir sous surveillance "
              "(vérifier qu'il ne se dégrade pas en contrepartie ci-dessus).")
    else:
        print("\n[VERDICT] Pas de gain net probant — le point faible F1_UP_FORT/F1_DOWN_FORT ne "
              "vient donc probablement pas d'une mauvaise calibration/seuil, mais bien d'un manque "
              "de signal exploitable par le modèle sur cette sous-tâche (cohérent avec l'échec des "
              "familles de features testées précédemment sur ce même point).")

    try:
        with pd.ExcelWriter('VIX_CALIBRATED_THRESHOLD_report.xlsx', engine='xlsxwriter') as w:
            agg.sort_values('delta_F1_UP_FORT', ascending=False).to_excel(w, 'Calibrated_vs_RAW', index=False)
            df_c.to_excel(w, 'Detail', index=False)
        print("\n[SAVE] VIX_CALIBRATED_THRESHOLD_report.xlsx (snapshot à date)")
    except Exception as e:
        print(f"[WARN Export] {e}")
else:
    print("Aucun résultat pour l'instant.")
print(f"\n[NOTE] {RESULTS_CSV} contient le détail complet — le recharger pour reprendre.")


In [ ]:
# ============================================================
# PUSH FINAL DU RAPPORT (xlsx) EN PLUS DU CSV DE PROGRESSION
# ============================================================
def push_report_file():
    if not GITHUB_TOKEN or not os.path.exists('VIX_CALIBRATED_THRESHOLD_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["cp", "VIX_CALIBRATED_THRESHOLD_report.xlsx",
                        f"{_PUSH_WORKDIR}/VIX_CALIBRATED_THRESHOLD_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_CALIBRATED_THRESHOLD_report.xlsx"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport Calibrated Threshold agrégé — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_CALIBRATED_THRESHOLD_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_progress(label='rapport final')
push_report_file()
